# Diffusion for Simulation

Toy conditional diffusion: generate straight vs left-turn trajectories conditioned on an intent bit.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
print(device)

In [ ]:
T, D, steps = 20, 40, 100
time = torch.linspace(0, 1, T, device=device)
betas = torch.linspace(1e-4, 0.02, steps, device=device)
alpha_bar = torch.cumprod(1 - betas, dim=0)

def sample_batch(batch):
    intent = torch.randint(0, 2, (batch, 1), device=device).float()
    straight = torch.stack([5 * time, torch.zeros_like(time)], dim=-1)
    left = torch.stack([2.5 * time, 4 * time], dim=-1)
    traj = torch.where(intent[:, None, :] == 0, straight[None], left[None])
    traj = traj + 0.03 * torch.randn_like(traj)
    return traj.reshape(batch, -1), intent

def q_sample(x0, t, noise):
    a = alpha_bar[t].view(-1, 1)
    return a.sqrt() * x0 + (1 - a).sqrt() * noise

In [ ]:
class CondDenoiser(nn.Module):
    def __init__(self):
        super().__init__()
        self.t = nn.Embedding(steps, 64)
        self.net = nn.Sequential(nn.Linear(D + 1 + 64, 128), nn.ReLU(), nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, D))
    def forward(self, xt, t, cond):
        return self.net(torch.cat([xt, cond, self.t(t)], dim=-1))

model = CondDenoiser().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
for step in range(1500):
    x0, cond = sample_batch(256)
    t = torch.randint(0, steps, (x0.size(0),), device=device)
    noise = torch.randn_like(x0)
    xt = q_sample(x0, t, noise)
    loss = F.mse_loss(model(xt, t, cond), noise)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 500 == 0:
        print(step, float(loss))

In [ ]:
# Simple DDPM-like reverse demo. Not production sampling, enough for intuition.
@torch.no_grad()
def sample(cond_value, n=8):
    x = torch.randn(n, D, device=device)
    cond = torch.full((n, 1), float(cond_value), device=device)
    for ti in reversed(range(steps)):
        t = torch.full((n,), ti, device=device, dtype=torch.long)
        eps = model(x, t, cond)
        a = alpha_bar[t].view(-1, 1)
        x0_hat = (x - (1 - a).sqrt() * eps) / a.sqrt().clamp_min(1e-6)
        if ti > 0:
            a_prev = alpha_bar[t - 1].view(-1, 1)
            x = a_prev.sqrt() * x0_hat + (1 - a_prev).sqrt() * eps
        else:
            x = x0_hat
    return x.reshape(n, T, 2).cpu()

plt.figure(figsize=(10, 4))
for panel, intent in enumerate([0, 1]):
    samples = sample(intent, n=8)
    plt.subplot(1, 2, panel + 1)
    for s in samples:
        plt.plot(s[:, 0], s[:, 1], marker='o', alpha=0.7)
    plt.title('straight intent' if intent == 0 else 'left intent')
    plt.axis('equal')
plt.show()